<a href="https://colab.research.google.com/github/samar8x/CVIP_Project/blob/main/Notebooks/06_Matched_ACDC_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Matched ACDC Normal/Adverse Analysis


This notebook performs the matched ACDC analysis using the same fixed Cityscapes-trained SegFormer-B2 model.

Two matched scene-level analyses are reported:

- **Primary matched analysis:** ACDC validation correspondences only — **203 labeled pairs**
- **Expanded robustness analysis:** ACDC train + validation correspondences — **1,003 labeled pairs**

The expanded analysis is included to improve support for rare semantic classes.

### Important interpretation of ACDC correspondence

ACDC normal-reference and adverse images correspond to the same general scene, but they are captured at different times.

Therefore, the correspondence provides stronger control over **static scene structure**, but it does **not** guarantee object-level matching for dynamic classes such as pedestrians, riders, bicycles, motorcycles, cars, buses, trucks, and trains.

For these dynamic classes, the normal and adverse images may contain different objects. Their matched results must therefore be interpreted as **scene-level normal-to-adverse comparisons**, not comparisons of identical object instances.

### Invalid / uncertainty masks

For the adverse image of every pair, pixels with:

- semantic label `255`, or
- ACDC `invIds == 1`

are excluded from the confusion matrix and IoU calculation.

The adverse invalid mask is **not projected onto the normal-reference image** because the two captures are not pixel-aligned.

### Additional outputs

This notebook also records, for every class and condition:

- number of matched scenes containing the class in the normal image,
- number containing it in the adverse image,
- number containing it in both,
- number containing it in either image.

These support counts are used to interpret rare-class results cautiously.

Mount Drive and define paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import zipfile
import shutil
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/CV_Project"
)

DATASETS_DIR = (
    PROJECT_ROOT / "datasets"
)

RESULTS_DIR = (
    PROJECT_ROOT / "results"
)

ACDC_RGB_ZIP = (
    DATASETS_DIR
    / "acdc"
    / "rgb_anon.zip"
)

ACDC_GT_ZIP = (
    DATASETS_DIR
    / "archives"
    / "gt_trainval.zip"
)

ACDC_GT_REF_ZIP = (
    DATASETS_DIR
    / "archives"
    / "gt_trainval_ref.zip"
)

MATCHED_RESULTS_DIR = (
    RESULTS_DIR
    / "matched_acdc"
)

MATCHED_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Paths ready.")

Mounted at /content/drive
Paths ready.


Copy ACDC archives to local Colab storage

In [ ]:
LOCAL_RGB_ZIP = Path(
    "/content/rgb_anon.zip"
)

LOCAL_GT_ZIP = Path(
    "/content/gt_trainval.zip"
)

LOCAL_GT_REF_ZIP = Path(
    "/content/gt_trainval_ref.zip"
)

archive_pairs = [
    (
        ACDC_RGB_ZIP,
        LOCAL_RGB_ZIP
    ),
    (
        ACDC_GT_ZIP,
        LOCAL_GT_ZIP
    ),
    (
        ACDC_GT_REF_ZIP,
        LOCAL_GT_REF_ZIP
    ),
]

for source, destination in archive_pairs:

    if not destination.exists():

        print(
            "Copying:",
            source.name
        )

        shutil.copy2(
            source,
            destination
        )

    else:

        print(
            "Local copy already exists:",
            destination.name
        )

print("Local archive copies ready.")

Copying: rgb_anon.zip
Copying: gt_trainval.zip
Copying: gt_trainval_ref.zip
Local archive copies ready.


Define conditions, splits, and filename helper

In [ ]:
conditions = [
    "fog",
    "rain",
    "snow",
    "night"
]

REFERENCE_SPLITS = [
    "train_ref",
    "val_ref"
]

ADVERSE_SPLITS = [
    "train",
    "val"
]


def get_scene_key(
    filename,
    suffix
):
    return (
        Path(filename)
        .name
        .replace(
            suffix,
            ""
        )
    )


def reference_to_adverse_split(
    reference_split
):

    if reference_split == "train_ref":
        return "train"

    if reference_split == "val_ref":
        return "val"

    raise ValueError(
        f"Unexpected reference split: "
        f"{reference_split}"
    )

Load all relevant archive filenames

In [ ]:
with zipfile.ZipFile(
    LOCAL_RGB_ZIP,
    "r"
) as z:

    rgb_names = [
        name
        for name in z.namelist()
        if "__MACOSX" not in name
    ]


with zipfile.ZipFile(
    LOCAL_GT_ZIP,
    "r"
) as z:

    adverse_gt_names = [
        name
        for name in z.namelist()
        if "__MACOSX" not in name
    ]


with zipfile.ZipFile(
    LOCAL_GT_REF_ZIP,
    "r"
) as z:

    reference_gt_names = [
        name
        for name in z.namelist()
        if "__MACOSX" not in name
    ]


print(
    "RGB archive entries:",
    len(rgb_names)
)

print(
    "Adverse GT archive entries:",
    len(adverse_gt_names)
)

print(
    "Reference GT archive entries:",
    len(reference_gt_names)
)

RGB archive entries: 4148
Adverse GT archive entries: 10032
Reference GT archive entries: 3012


Build RGB, semantic-GT, and invalid-mask lookups

In [ ]:
ref_rgb_lookup = {}
adverse_rgb_lookup = {}
ref_gt_lookup = {}
adverse_gt_lookup = {}
adverse_invalid_lookup = {}


# ------------------------------------------------------------
# RGB lookups
# ------------------------------------------------------------

for name in rgb_names:

    if not name.endswith(
        "_rgb_ref_anon.png"
    ):
        continue

    parts = name.split("/")

    if len(parts) < 5:
        continue

    condition = parts[1]
    split = parts[2]

    if (
        condition not in conditions
        or split not in REFERENCE_SPLITS
    ):
        continue

    key = get_scene_key(
        name,
        "_rgb_ref_anon.png"
    )

    ref_rgb_lookup[
        (
            condition,
            split,
            key
        )
    ] = name


for name in rgb_names:

    if not name.endswith(
        "_rgb_anon.png"
    ):
        continue

    parts = name.split("/")

    if len(parts) < 5:
        continue

    condition = parts[1]
    split = parts[2]

    if (
        condition not in conditions
        or split not in ADVERSE_SPLITS
    ):
        continue

    key = get_scene_key(
        name,
        "_rgb_anon.png"
    )

    adverse_rgb_lookup[
        (
            condition,
            split,
            key
        )
    ] = name


# ------------------------------------------------------------
# Reference GT lookup
# ------------------------------------------------------------

for name in reference_gt_names:

    if not name.endswith(
        "_gt_ref_labelTrainIds.png"
    ):
        continue

    parts = name.split("/")

    if len(parts) < 5:
        continue

    condition = parts[1]
    split = parts[2]

    if (
        condition not in conditions
        or split not in REFERENCE_SPLITS
    ):
        continue

    key = get_scene_key(
        name,
        "_gt_ref_labelTrainIds.png"
    )

    ref_gt_lookup[
        (
            condition,
            split,
            key
        )
    ] = name


# ------------------------------------------------------------
# Adverse semantic GT + invIds
# ------------------------------------------------------------

for name in adverse_gt_names:

    parts = name.split("/")

    if len(parts) < 5:
        continue

    condition = parts[1]
    split = parts[2]

    if (
        condition not in conditions
        or split not in ADVERSE_SPLITS
    ):
        continue

    if name.endswith(
        "_gt_labelTrainIds.png"
    ):

        key = get_scene_key(
            name,
            "_gt_labelTrainIds.png"
        )

        adverse_gt_lookup[
            (
                condition,
                split,
                key
            )
        ] = name

    elif name.endswith(
        "_gt_invIds.png"
    ):

        key = get_scene_key(
            name,
            "_gt_invIds.png"
        )

        adverse_invalid_lookup[
            (
                condition,
                split,
                key
            )
        ] = name


print(
    "Reference RGB lookup:",
    len(ref_rgb_lookup)
)

print(
    "Adverse RGB lookup:",
    len(adverse_rgb_lookup)
)

print(
    "Reference GT lookup:",
    len(ref_gt_lookup)
)

print(
    "Adverse GT lookup:",
    len(adverse_gt_lookup)
)

print(
    "Adverse invIds lookup:",
    len(adverse_invalid_lookup)
)

Reference RGB lookup: 2006
Adverse RGB lookup: 2006
Reference GT lookup: 1003
Adverse GT lookup: 2006
Adverse invIds lookup: 2006


Build and verify the expanded 1,003-pair manifest

In [ ]:
matched_rows = []

missing_ref_rgb = []
missing_adverse_rgb = []
missing_adverse_gt = []
missing_adverse_invalid = []


for (
    condition,
    ref_split,
    key
), ref_gt_path in ref_gt_lookup.items():

    adverse_split = (
        reference_to_adverse_split(
            ref_split
        )
    )

    ref_key = (
        condition,
        ref_split,
        key
    )

    adverse_key = (
        condition,
        adverse_split,
        key
    )

    ref_rgb_path = (
        ref_rgb_lookup.get(
            ref_key
        )
    )

    adverse_rgb_path = (
        adverse_rgb_lookup.get(
            adverse_key
        )
    )

    adverse_gt_path = (
        adverse_gt_lookup.get(
            adverse_key
        )
    )

    adverse_invalid_path = (
        adverse_invalid_lookup.get(
            adverse_key
        )
    )

    if ref_rgb_path is None:
        missing_ref_rgb.append(
            ref_key
        )

    if adverse_rgb_path is None:
        missing_adverse_rgb.append(
            adverse_key
        )

    if adverse_gt_path is None:
        missing_adverse_gt.append(
            adverse_key
        )

    if adverse_invalid_path is None:
        missing_adverse_invalid.append(
            adverse_key
        )

    if (
        ref_rgb_path is not None
        and adverse_rgb_path is not None
        and adverse_gt_path is not None
        and adverse_invalid_path is not None
    ):

        matched_rows.append({
            "condition":
                condition,

            "reference_split":
                ref_split,

            "adverse_split":
                adverse_split,

            "scene_key":
                key,

            "ref_rgb":
                ref_rgb_path,

            "ref_gt":
                ref_gt_path,

            "adverse_rgb":
                adverse_rgb_path,

            "adverse_gt":
                adverse_gt_path,

            "adverse_invalid":
                adverse_invalid_path,
        })


expanded_pairs_df = pd.DataFrame(
    matched_rows
)

primary_pairs_df = (
    expanded_pairs_df[
        expanded_pairs_df[
            "reference_split"
        ] == "val_ref"
    ]
    .reset_index(drop=True)
)


print(
    "Expanded matched pairs:",
    len(expanded_pairs_df)
)

print(
    "Primary validation pairs:",
    len(primary_pairs_df)
)

print(
    "Missing reference RGB:",
    len(missing_ref_rgb)
)

print(
    "Missing adverse RGB:",
    len(missing_adverse_rgb)
)

print(
    "Missing adverse GT:",
    len(missing_adverse_gt)
)

print(
    "Missing adverse invIds:",
    len(missing_adverse_invalid)
)

assert len(expanded_pairs_df) == 1003
assert len(primary_pairs_df) == 203

assert len(missing_ref_rgb) == 0
assert len(missing_adverse_rgb) == 0
assert len(missing_adverse_gt) == 0
assert len(missing_adverse_invalid) == 0

print(
    "✓ All labeled matched scene pairs verified."
)

Expanded matched pairs: 1003
Primary validation pairs: 203
Missing reference RGB: 0
Missing adverse RGB: 0
Missing adverse GT: 0
Missing adverse invIds: 0
✓ All labeled matched scene pairs verified.


Count primary and expanded pairs by condition

In [ ]:
primary_counts = (
    primary_pairs_df[
        "condition"
    ]
    .value_counts()
    .reindex(
        conditions
    )
)

expanded_counts = (
    expanded_pairs_df[
        "condition"
    ]
    .value_counts()
    .reindex(
        conditions
    )
)

pair_count_table = pd.DataFrame({
    "primary_val_pairs":
        primary_counts,

    "expanded_train_val_pairs":
        expanded_counts
})

display(
    pair_count_table
)

,primary_val_pairs,expanded_train_val_pairs
condition,,
fog,50,250
rain,50,250
snow,50,250
night,53,253


Save both matched-pair manifests

In [ ]:
primary_manifest_path = (
    MATCHED_RESULTS_DIR
    / "matched_pairs_manifest_primary_203.csv"
)

expanded_manifest_path = (
    MATCHED_RESULTS_DIR
    / "matched_pairs_manifest_expanded_1003.csv"
)

primary_pairs_df.to_csv(
    primary_manifest_path,
    index=False
)

expanded_pairs_df.to_csv(
    expanded_manifest_path,
    index=False
)

print(
    "Primary manifest:",
    primary_manifest_path
)

print(
    "Expanded manifest:",
    expanded_manifest_path
)

Primary manifest: /content/drive/MyDrive/CV_Project/results/matched_acdc/matched_pairs_manifest_primary_203.csv
Expanded manifest: /content/drive/MyDrive/CV_Project/results/matched_acdc/matched_pairs_manifest_expanded_1003.csv


## Matching Limitation for Dynamic Classes

The ACDC correspondence is a **scene-level correspondence**.

Static structures such as roads, buildings, signs, and poles are more naturally controlled by the paired-scene design.

Dynamic objects may differ between the two captures because the normal and adverse images were recorded at different times.

The following classes are therefore flagged as dynamic in later support tables:

- person
- rider
- car
- truck
- bus
- train
- motorcycle
- bicycle

For these classes, a matched IoU change may reflect both adverse-condition effects and changes in scene content.

Prepare local folders for the 1,003 labeled pairs

In [ ]:
LOCAL_MATCHED_ROOT = Path(
    "/content/matched_acdc"
)

REF_RGB_DIR = (
    LOCAL_MATCHED_ROOT
    / "ref_rgb"
)

REF_GT_DIR = (
    LOCAL_MATCHED_ROOT
    / "ref_gt"
)

ADV_RGB_DIR = (
    LOCAL_MATCHED_ROOT
    / "adverse_rgb"
)

ADV_GT_DIR = (
    LOCAL_MATCHED_ROOT
    / "adverse_gt"
)

ADV_INVALID_DIR = (
    LOCAL_MATCHED_ROOT
    / "adverse_invalid"
)

for folder in [
    REF_RGB_DIR,
    REF_GT_DIR,
    ADV_RGB_DIR,
    ADV_GT_DIR,
    ADV_INVALID_DIR,
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Local matched-data folders ready."
)

Local matched-data folders ready.


Extract the files required by the expanded manifest

In [ ]:
def extract_unique_paths(
    zip_path,
    archive_paths,
    output_root,
    label
):

    unique_paths = list(
        dict.fromkeys(
            archive_paths
        )
    )

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        for i, path in enumerate(
            unique_paths,
            start=1
        ):

            z.extract(
                path,
                output_root
            )

            if (
                i % 250 == 0
                or i == len(unique_paths)
            ):

                print(
                    f"{label}: "
                    f"{i}/{len(unique_paths)}"
                )


extract_unique_paths(
    LOCAL_RGB_ZIP,
    expanded_pairs_df[
        "ref_rgb"
    ],
    REF_RGB_DIR,
    "Reference RGB"
)

extract_unique_paths(
    LOCAL_RGB_ZIP,
    expanded_pairs_df[
        "adverse_rgb"
    ],
    ADV_RGB_DIR,
    "Adverse RGB"
)

extract_unique_paths(
    LOCAL_GT_REF_ZIP,
    expanded_pairs_df[
        "ref_gt"
    ],
    REF_GT_DIR,
    "Reference GT"
)

extract_unique_paths(
    LOCAL_GT_ZIP,
    expanded_pairs_df[
        "adverse_gt"
    ],
    ADV_GT_DIR,
    "Adverse GT"
)

extract_unique_paths(
    LOCAL_GT_ZIP,
    expanded_pairs_df[
        "adverse_invalid"
    ],
    ADV_INVALID_DIR,
    "Adverse invIds"
)

print(
    "Expanded matched files extracted."
)

Reference RGB: 250/1003
Reference RGB: 500/1003
Reference RGB: 750/1003
Reference RGB: 1000/1003
Reference RGB: 1003/1003
Adverse RGB: 250/1003
Adverse RGB: 500/1003
Adverse RGB: 750/1003
Adverse RGB: 1000/1003
Adverse RGB: 1003/1003
Reference GT: 250/1003
Reference GT: 500/1003
Reference GT: 750/1003
Reference GT: 1000/1003
Reference GT: 1003/1003
Adverse GT: 250/1003
Adverse GT: 500/1003
Adverse GT: 750/1003
Adverse GT: 1000/1003
Adverse GT: 1003/1003
Adverse invIds: 250/1003
Adverse invIds: 500/1003
Adverse invIds: 750/1003
Adverse invIds: 1000/1003
Adverse invIds: 1003/1003
Expanded matched files extracted.


Install/import model libraries

In [ ]:
!pip -q install transformers

In [ ]:
import torch
import torch.nn.functional as F

from PIL import Image

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation
)

from tqdm.auto import tqdm

print(
    "PyTorch:",
    torch.__version__
)

PyTorch: 2.11.0+cu130


Load the same fixed SegFormer-B2

In [ ]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

MODEL_NAME = (
    "nvidia/"
    "segformer-b2-finetuned-cityscapes-1024-1024"
)

processor = (
    SegformerImageProcessor
    .from_pretrained(
        MODEL_NAME
    )
)

model = (
    SegformerForSemanticSegmentation
    .from_pretrained(
        MODEL_NAME
    )
)

model = model.to(
    DEVICE
)

model.eval()

print(
    "Device:",
    DEVICE
)

print(
    "Model:",
    MODEL_NAME
)

preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.68k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  110MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

Device: cuda
Model: nvidia/segformer-b2-finetuned-cityscapes-1024-1024


Define classes and evaluation helpers

In [ ]:
CLASS_NAMES = [
    "road",
    "sidewalk",
    "building",
    "wall",
    "fence",
    "pole",
    "traffic light",
    "traffic sign",
    "vegetation",
    "terrain",
    "sky",
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motorcycle",
    "bicycle"
]

NUM_CLASSES = 19
IGNORE_INDEX = 255

DYNAMIC_CLASSES = [
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motorcycle",
    "bicycle"
]

Prediction helper

In [ ]:
def predict_mask(
    image,
    target_shape
):

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    pixel_values = (
        inputs[
            "pixel_values"
        ]
        .to(
            DEVICE
        )
    )

    with torch.inference_mode():

        outputs = model(
            pixel_values=pixel_values
        )

    logits = F.interpolate(
        outputs.logits,
        size=target_shape,
        mode="bilinear",
        align_corners=False
    )

    prediction = (
        logits
        .argmax(
            dim=1
        )[0]
        .cpu()
        .numpy()
    )

    return prediction

Confusion matrix helper with explicit ACDC `invIds` handling

In [ ]:
def confusion_from_arrays(
    ground_truth,
    prediction,
    invalid_mask=None
):

    valid = (
        (ground_truth != IGNORE_INDEX)
        & (ground_truth >= 0)
        & (ground_truth < NUM_CLASSES)
        & (prediction >= 0)
        & (prediction < NUM_CLASSES)
    )

    if invalid_mask is not None:

        if (
            invalid_mask.shape
            != ground_truth.shape
        ):

            raise ValueError(
                "GT and invIds shape mismatch: "
                f"{ground_truth.shape} vs "
                f"{invalid_mask.shape}"
            )

        # ACDC invIds:
        # 0 = valid
        # 1 = invalid / uncertain
        valid &= (
            invalid_mask == 0
        )

    gt = (
        ground_truth[
            valid
        ]
        .astype(
            np.int64
        )
    )

    pred = (
        prediction[
            valid
        ]
        .astype(
            np.int64
        )
    )

    encoded = (
        NUM_CLASSES
        * gt
        + pred
    )

    return (
        np.bincount(
            encoded,
            minlength=NUM_CLASSES ** 2
        )
        .reshape(
            NUM_CLASSES,
            NUM_CLASSES
        )
    )

IoU and class-presence helpers

In [ ]:
def calculate_iou(
    confusion_matrix
):

    intersection = (
        np.diag(
            confusion_matrix
        )
        .astype(
            np.float64
        )
    )

    gt_total = (
        confusion_matrix
        .sum(
            axis=1
        )
        .astype(
            np.float64
        )
    )

    pred_total = (
        confusion_matrix
        .sum(
            axis=0
        )
        .astype(
            np.float64
        )
    )

    union = (
        gt_total
        + pred_total
        - intersection
    )

    iou = np.full(
        NUM_CLASSES,
        np.nan,
        dtype=np.float64
    )

    valid = (
        union > 0
    )

    iou[
        valid
    ] = (
        intersection[
            valid
        ]
        / union[
            valid
        ]
    )

    return iou


def class_presence(
    ground_truth,
    invalid_mask=None
):

    valid = (
        (ground_truth != IGNORE_INDEX)
        & (ground_truth >= 0)
        & (ground_truth < NUM_CLASSES)
    )

    if invalid_mask is not None:

        valid &= (
            invalid_mask == 0
        )

    presence = np.zeros(
        NUM_CLASSES,
        dtype=bool
    )

    for class_id in range(
        NUM_CLASSES
    ):

        presence[
            class_id
        ] = np.any(
            valid
            & (
                ground_truth
                == class_id
            )
        )

    return presence

Define local-path helper

In [ ]:
def local_extracted_path(
    root_folder,
    archive_path
):

    return (
        root_folder
        / archive_path
    )

Prepare masked per-pair cache

In [ ]:
PER_PAIR_DIR = (
    MATCHED_RESULTS_DIR
    / "per_pair_confusions_masked_v2"
)

PER_PAIR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Per-pair cache:"
)

print(
    PER_PAIR_DIR
)

Per-pair cache:
/content/drive/MyDrive/CV_Project/results/matched_acdc/per_pair_confusions_masked_v2


## Evaluate all 1,003 labeled scene pairs once

Each `.npz` file stores:

- normal-reference 19 × 19 confusion matrix,
- adverse 19 × 19 confusion matrix with `invIds == 1` excluded,
- normal class-presence flags,
- adverse class-presence flags,
- split and scene identifiers.

The validation-only primary analysis is later obtained by selecting the `val_ref` pair files, so the model does not need to be run twice.

In [ ]:
for row in tqdm(
    expanded_pairs_df.itertuples(
        index=False
    ),
    total=len(
        expanded_pairs_df
    ),
    desc="Evaluating matched ACDC pairs"
):

    cache_name = (
        f"{row.condition}"
        f"__{row.reference_split}"
        f"__{row.scene_key}.npz"
    )

    cache_path = (
        PER_PAIR_DIR
        / cache_name
    )

    if cache_path.exists():
        continue

    # --------------------------------------------------------
    # Local file paths
    # --------------------------------------------------------

    ref_rgb_path = (
        local_extracted_path(
            REF_RGB_DIR,
            row.ref_rgb
        )
    )

    ref_gt_path = (
        local_extracted_path(
            REF_GT_DIR,
            row.ref_gt
        )
    )

    adverse_rgb_path = (
        local_extracted_path(
            ADV_RGB_DIR,
            row.adverse_rgb
        )
    )

    adverse_gt_path = (
        local_extracted_path(
            ADV_GT_DIR,
            row.adverse_gt
        )
    )

    adverse_invalid_path = (
        local_extracted_path(
            ADV_INVALID_DIR,
            row.adverse_invalid
        )
    )

    # --------------------------------------------------------
    # Load images and masks
    # --------------------------------------------------------

    ref_image = Image.open(
        ref_rgb_path
    ).convert("RGB")

    adverse_image = Image.open(
        adverse_rgb_path
    ).convert("RGB")

    ref_gt = np.array(
        Image.open(
            ref_gt_path
        )
    )

    adverse_gt = np.array(
        Image.open(
            adverse_gt_path
        )
    )

    adverse_invalid = np.array(
        Image.open(
            adverse_invalid_path
        )
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    ref_prediction = predict_mask(
        ref_image,
        ref_gt.shape
    )

    adverse_prediction = predict_mask(
        adverse_image,
        adverse_gt.shape
    )

    # --------------------------------------------------------
    # Per-pair confusion matrices
    # --------------------------------------------------------

    ref_confusion = (
        confusion_from_arrays(
            ref_gt,
            ref_prediction,
            invalid_mask=None
        )
    )

    adverse_confusion = (
        confusion_from_arrays(
            adverse_gt,
            adverse_prediction,
            invalid_mask=
                adverse_invalid
        )
    )

    # --------------------------------------------------------
    # Class presence
    # --------------------------------------------------------

    ref_presence = (
        class_presence(
            ref_gt
        )
    )

    adverse_presence = (
        class_presence(
            adverse_gt,
            invalid_mask=
                adverse_invalid
        )
    )

    np.savez_compressed(
        cache_path,

        condition=
            row.condition,

        reference_split=
            row.reference_split,

        adverse_split=
            row.adverse_split,

        scene_key=
            row.scene_key,

        ref_confusion=
            ref_confusion,

        adverse_confusion=
            adverse_confusion,

        ref_presence=
            ref_presence,

        adverse_presence=
            adverse_presence
    )


print(
    "Cached pair files:",
    len(
        list(
            PER_PAIR_DIR.glob(
                "*.npz"
            )
        )
    )
)

Evaluating matched ACDC pairs:   0%|          | 0/1003 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  109MB            

model.safetensors: downloading bytes:           |  0.00B            

Cached pair files: 1003


Verify cache counts

In [ ]:
cache_rows = []

for file_path in sorted(
    PER_PAIR_DIR.glob(
        "*.npz"
    )
):

    data = np.load(
        file_path,
        allow_pickle=True
    )

    cache_rows.append({
        "file":
            file_path,

        "condition":
            str(
                data[
                    "condition"
                ]
            ),

        "reference_split":
            str(
                data[
                    "reference_split"
                ]
            ),

        "scene_key":
            str(
                data[
                    "scene_key"
                ]
            ),
    })


cache_df = pd.DataFrame(
    cache_rows
)

print(
    "Total cached pairs:",
    len(cache_df)
)

display(
    pd.crosstab(
        cache_df[
            "condition"
        ],
        cache_df[
            "reference_split"
        ]
    )
)

assert len(cache_df) == 1003

print(
    "✓ Expanded pair cache verified."
)

Total cached pairs: 1003


reference_split,train_ref,val_ref
condition,,
fog,200,50
night,200,53
rain,200,50
snow,200,50


✓ Expanded pair cache verified.


Aggregate a selected analysis set

In [ ]:
def aggregate_analysis_set(
    pair_cache_df
):

    results = {}

    for condition in conditions:

        condition_files = (
            pair_cache_df[
                pair_cache_df[
                    "condition"
                ] == condition
            ][
                "file"
            ]
            .tolist()
        )

        ref_confusion = np.zeros(
            (
                NUM_CLASSES,
                NUM_CLASSES
            ),
            dtype=np.int64
        )

        adverse_confusion = np.zeros(
            (
                NUM_CLASSES,
                NUM_CLASSES
            ),
            dtype=np.int64
        )

        ref_presence_rows = []
        adverse_presence_rows = []

        for file_path in condition_files:

            data = np.load(
                file_path,
                allow_pickle=True
            )

            ref_confusion += (
                data[
                    "ref_confusion"
                ]
            )

            adverse_confusion += (
                data[
                    "adverse_confusion"
                ]
            )

            ref_presence_rows.append(
                data[
                    "ref_presence"
                ]
            )

            adverse_presence_rows.append(
                data[
                    "adverse_presence"
                ]
            )

        ref_presence_array = (
            np.stack(
                ref_presence_rows
            )
        )

        adverse_presence_array = (
            np.stack(
                adverse_presence_rows
            )
        )

        ref_iou = calculate_iou(
            ref_confusion
        )

        adverse_iou = calculate_iou(
            adverse_confusion
        )

        ref_miou = np.nanmean(
            ref_iou
        )

        adverse_miou = np.nanmean(
            adverse_iou
        )

        matched_change = (
            ref_iou
            - adverse_iou
        )

        matched_miou_change = (
            ref_miou
            - adverse_miou
        )

        results[
            condition
        ] = {
            "num_pairs":
                len(
                    condition_files
                ),

            "ref_confusion":
                ref_confusion,

            "adverse_confusion":
                adverse_confusion,

            "ref_iou":
                ref_iou,

            "adverse_iou":
                adverse_iou,

            "matched_change":
                matched_change,

            "ref_miou":
                ref_miou,

            "adverse_miou":
                adverse_miou,

            "matched_miou_change":
                matched_miou_change,

            "ref_presence":
                ref_presence_array,

            "adverse_presence":
                adverse_presence_array,
        }

    return results

Create primary 203-pair and expanded 1,003-pair result sets

In [ ]:
primary_cache_df = (
    cache_df[
        cache_df[
            "reference_split"
        ] == "val_ref"
    ]
    .reset_index(drop=True)
)

expanded_cache_df = (
    cache_df.copy()
)


primary_results = (
    aggregate_analysis_set(
        primary_cache_df
    )
)

expanded_results = (
    aggregate_analysis_set(
        expanded_cache_df
    )
)


print(
    "Primary cached pairs:",
    len(primary_cache_df)
)

print(
    "Expanded cached pairs:",
    len(expanded_cache_df)
)

Primary cached pairs: 203
Expanded cached pairs: 1003


Build condition-level mIoU summaries

In [ ]:
def build_condition_summary(
    results,
    analysis_set
):

    rows = []

    for condition in conditions:

        result = (
            results[
                condition
            ]
        )

        rows.append({
            "analysis_set":
                analysis_set,

            "condition":
                condition,

            "num_pairs":
                result[
                    "num_pairs"
                ],

            "normal_ref_mIoU":
                result[
                    "ref_miou"
                ],

            "adverse_mIoU":
                result[
                    "adverse_miou"
                ],

            "matched_mIoU_change":
                result[
                    "matched_miou_change"
                ],
        })

    return pd.DataFrame(
        rows
    )


primary_summary_df = (
    build_condition_summary(
        primary_results,
        "primary_validation_203"
    )
)

expanded_summary_df = (
    build_condition_summary(
        expanded_results,
        "expanded_train_val_1003"
    )
)


print(
    "Primary validation analysis"
)

display(
    primary_summary_df
)

print(
    "Expanded robustness analysis"
)

display(
    expanded_summary_df
)

Primary validation analysis


,analysis_set,condition,num_pairs,normal_ref_mIoU,adverse_mIoU,matched_mIoU_change
0,primary_validation_203,fog,50,0.521605,0.654478,-0.132872
1,primary_validation_203,rain,50,0.561446,0.548373,0.013073
2,primary_validation_203,snow,50,0.509152,0.502509,0.006643
3,primary_validation_203,night,53,0.498239,0.350868,0.147371


Expanded robustness analysis


,analysis_set,condition,num_pairs,normal_ref_mIoU,adverse_mIoU,matched_mIoU_change
0,expanded_train_val_1003,fog,250,0.566538,0.636456,-0.069917
1,expanded_train_val_1003,rain,250,0.577430,0.535404,0.042026
2,expanded_train_val_1003,snow,250,0.607713,0.511846,0.095867
3,expanded_train_val_1003,night,253,0.564162,0.386916,0.177246


Build per-class matched results

In [ ]:
def build_per_class_table(
    results,
    analysis_set
):

    table = pd.DataFrame({
        "class_id":
            list(
                range(
                    NUM_CLASSES
                )
            ),

        "class_name":
            CLASS_NAMES,
    })

    table[
        "analysis_set"
    ] = analysis_set

    for condition in conditions:

        result = (
            results[
                condition
            ]
        )

        table[
            f"{condition}_normal_iou"
        ] = result[
            "ref_iou"
        ]

        table[
            f"{condition}_adverse_iou"
        ] = result[
            "adverse_iou"
        ]

        table[
            f"{condition}_change"
        ] = result[
            "matched_change"
        ]

    return table


primary_per_class_df = (
    build_per_class_table(
        primary_results,
        "primary_validation_203"
    )
)

expanded_per_class_df = (
    build_per_class_table(
        expanded_results,
        "expanded_train_val_1003"
    )
)


print(
    "Primary per-class results"
)

display(
    primary_per_class_df
)

print(
    "Expanded per-class results"
)

display(
    expanded_per_class_df
)

Primary per-class results


,class_id,class_name,analysis_set,fog_normal_iou,fog_adverse_iou,fog_change,rain_normal_iou,rain_adverse_iou,rain_change,snow_normal_iou,snow_adverse_iou,snow_change,night_normal_iou,night_adverse_iou,night_change
0,0,road,primary_validation_203,0.890839,0.936080,-0.045241,0.857482,0.825257,0.032226,0.862198,0.759230,0.102968,0.801105,0.776748,0.024357
1,1,sidewalk,primary_validation_203,0.651435,0.729297,-0.077862,0.506580,0.333073,0.173507,0.431983,0.264574,0.167409,0.177573,0.188830,-0.011257
2,2,building,primary_validation_203,0.868934,0.813972,0.054962,0.852519,0.859756,-0.007237,0.800269,0.748032,0.052237,0.835947,0.819351,0.016596
3,3,wall,primary_validation_203,0.538241,0.652662,-0.114420,0.335944,0.188953,0.146992,0.479002,0.287206,0.191796,0.458143,0.331308,0.126835
4,4,fence,primary_validation_203,0.484903,0.456511,0.028393,0.425606,0.274598,0.151009,0.309362,0.360162,-0.050800,0.452136,0.305395,0.146741
5,5,pole,primary_validation_203,0.529941,0.565047,-0.035106,0.475154,0.434187,0.040967,0.396562,0.372600,0.023962,0.419786,0.459958,-0.040172
6,6,traffic light,primary_validation_203,0.671137,0.737631,-0.066494,0.599929,0.646299,-0.046370,0.612297,0.633432,-0.021135,0.752882,0.507829,0.245053
7,7,traffic sign,primary_validation_203,0.514019,0.577510,-0.063491,0.563275,0.603141,-0.039866,0.532512,0.565638,-0.033126,0.435475,0.259417,0.176058
8,8,vegetation,primary_validation_203,0.857924,0.856692,0.001232,0.866508,0.886668,-0.020160,0.823728,0.811094,0.012635,0.860272,0.723381,0.136891
9,9,terrain,primary_validation_203,0.461654,0.512605,-0.050950,0.524148,0.515697,0.008451,0.416034,0.067376,0.348658,0.369489,0.196253,0.173236


Expanded per-class results


,class_id,class_name,analysis_set,fog_normal_iou,fog_adverse_iou,fog_change,rain_normal_iou,rain_adverse_iou,rain_change,snow_normal_iou,snow_adverse_iou,snow_change,night_normal_iou,night_adverse_iou,night_change
0,0,road,expanded_train_val_1003,0.914705,0.919863,-0.005159,0.860165,0.793767,0.066398,0.860786,0.769994,0.090792,0.851704,0.833498,0.018205
1,1,sidewalk,expanded_train_val_1003,0.516490,0.543570,-0.027080,0.572263,0.257888,0.314375,0.473491,0.280101,0.193389,0.394110,0.311252,0.082857
2,2,building,expanded_train_val_1003,0.838309,0.733462,0.104847,0.867601,0.853068,0.014533,0.830447,0.807725,0.022721,0.857362,0.822301,0.035061
3,3,wall,expanded_train_val_1003,0.405843,0.543077,-0.137234,0.536456,0.355657,0.180799,0.420146,0.338296,0.081850,0.469522,0.289349,0.180173
4,4,fence,expanded_train_val_1003,0.277543,0.216284,0.061259,0.363307,0.216931,0.146376,0.315333,0.407264,-0.091931,0.396339,0.265789,0.130549
5,5,pole,expanded_train_val_1003,0.456359,0.469536,-0.013177,0.498782,0.488673,0.010109,0.461852,0.460619,0.001233,0.419163,0.426937,-0.007774
6,6,traffic light,expanded_train_val_1003,0.491501,0.520365,-0.028864,0.629936,0.611191,0.018745,0.630791,0.650889,-0.020098,0.541656,0.447129,0.094527
7,7,traffic sign,expanded_train_val_1003,0.652335,0.635714,0.016621,0.561021,0.521954,0.039066,0.488713,0.579390,-0.090677,0.452135,0.278795,0.173340
8,8,vegetation,expanded_train_val_1003,0.874171,0.837127,0.037044,0.863384,0.881893,-0.018509,0.818413,0.795593,0.022820,0.861114,0.683313,0.177801
9,9,terrain,expanded_train_val_1003,0.629033,0.619922,0.009112,0.542004,0.559469,-0.017465,0.561904,0.071491,0.490413,0.464162,0.253448,0.210714


### Matched IoU change

For both analysis sets:

**Matched IoU change = Normal-reference IoU − Adverse IoU**

- positive value → lower measured IoU on the adverse side,
- negative value → higher measured IoU on the adverse side.

For dynamic classes, this quantity is a **scene-level matched change**, not an object-level causal weather effect.

Build class-support tables

In [ ]:
def build_support_table(
    results,
    analysis_set
):

    rows = []

    for condition in conditions:

        result = (
            results[
                condition
            ]
        )

        ref_presence = (
            result[
                "ref_presence"
            ]
        )

        adverse_presence = (
            result[
                "adverse_presence"
            ]
        )

        total_pairs = (
            result[
                "num_pairs"
            ]
        )

        for class_id, class_name in enumerate(
            CLASS_NAMES
        ):

            normal_count = int(
                ref_presence[
                    :,
                    class_id
                ].sum()
            )

            adverse_count = int(
                adverse_presence[
                    :,
                    class_id
                ].sum()
            )

            both_count = int(
                (
                    ref_presence[
                        :,
                        class_id
                    ]
                    &
                    adverse_presence[
                        :,
                        class_id
                    ]
                ).sum()
            )

            either_count = int(
                (
                    ref_presence[
                        :,
                        class_id
                    ]
                    |
                    adverse_presence[
                        :,
                        class_id
                    ]
                ).sum()
            )

            rows.append({
                "analysis_set":
                    analysis_set,

                "condition":
                    condition,

                "class_id":
                    class_id,

                "class_name":
                    class_name,

                "dynamic_class":
                    (
                        class_name
                        in DYNAMIC_CLASSES
                    ),

                "total_pairs":
                    total_pairs,

                "pairs_with_class_normal":
                    normal_count,

                "pairs_with_class_adverse":
                    adverse_count,

                "pairs_with_class_both":
                    both_count,

                "pairs_with_class_either":
                    either_count,

                # Descriptive warning only.
                # This is not a formal statistical threshold.
                "low_pair_support":
                    both_count < 10,
            })

    return pd.DataFrame(
        rows
    )


primary_support_df = (
    build_support_table(
        primary_results,
        "primary_validation_203"
    )
)

expanded_support_df = (
    build_support_table(
        expanded_results,
        "expanded_train_val_1003"
    )
)


print(
    "Primary validation support"
)

display(
    primary_support_df
)

print(
    "Expanded support for thin/small classes"
)

display(
    expanded_support_df[
        expanded_support_df[
            "class_name"
        ].isin([
            "pole",
            "traffic light",
            "traffic sign",
            "person",
            "rider",
            "motorcycle",
            "bicycle",
        ])
    ]
)

Primary validation support


,analysis_set,condition,class_id,class_name,dynamic_class,total_pairs,pairs_with_class_normal,pairs_with_class_adverse,pairs_with_class_both,pairs_with_class_either,low_pair_support
0,primary_validation_203,fog,0,road,False,50,50,50,50,50,False
1,primary_validation_203,fog,1,sidewalk,False,50,46,48,46,48,False
2,primary_validation_203,fog,2,building,False,50,48,47,47,48,False
3,primary_validation_203,fog,3,wall,False,50,37,26,23,40,False
4,primary_validation_203,fog,4,fence,False,50,33,26,23,36,False
...,...,...,...,...,...,...,...,...,...,...,...
71,primary_validation_203,night,14,truck,True,53,0,4,0,4,True
72,primary_validation_203,night,15,bus,True,53,0,0,0,0,True
73,primary_validation_203,night,16,train,True,53,16,10,5,21,True
74,primary_validation_203,night,17,motorcycle,True,53,15,6,2,19,True


Expanded support for thin/small classes


,analysis_set,condition,class_id,class_name,dynamic_class,total_pairs,pairs_with_class_normal,pairs_with_class_adverse,pairs_with_class_both,pairs_with_class_either,low_pair_support
5,expanded_train_val_1003,fog,5,pole,False,250,249,250,249,250,False
6,expanded_train_val_1003,fog,6,traffic light,False,250,94,88,79,103,False
7,expanded_train_val_1003,fog,7,traffic sign,False,250,224,197,190,231,False
11,expanded_train_val_1003,fog,11,person,True,250,87,52,37,102,False
12,expanded_train_val_1003,fog,12,rider,True,250,55,13,8,60,True
17,expanded_train_val_1003,fog,17,motorcycle,True,250,45,15,9,51,True
18,expanded_train_val_1003,fog,18,bicycle,True,250,40,17,8,49,True
24,expanded_train_val_1003,rain,5,pole,False,250,249,248,247,250,False
25,expanded_train_val_1003,rain,6,traffic light,False,250,127,123,116,134,False
26,expanded_train_val_1003,rain,7,traffic sign,False,250,233,231,229,235,False


## Rare-Class Sample-Size Interpretation

The expanded 1,003-pair analysis is included specifically to improve support for rare classes.

However, the effective sample size for a class is not simply the total number of scene pairs. It depends on how many pairs actually contain that class.

The support tables therefore report the number of pairs in which each class appears in:

- the normal image,
- the adverse image,
- both images,
- either image.

Classes with very low `pairs_with_class_both` should be interpreted cautiously even in the expanded analysis.

The `low_pair_support` flag uses `< 10` pairs only as a descriptive warning in this project; it is **not** treated as a formal statistical cutoff.

Show the five largest measured matched changes

In [ ]:
def show_top_changes(
    table,
    title
):

    print(
        "\n" + title
    )

    for condition in conditions:

        print(
            "\n" + "=" * 60
        )

        print(
            condition.upper()
        )

        print(
            "=" * 60
        )

        top5 = (
            table[
                [
                    "class_name",
                    f"{condition}_normal_iou",
                    f"{condition}_adverse_iou",
                    f"{condition}_change",
                ]
            ]
            .sort_values(
                f"{condition}_change",
                ascending=False
            )
            .head(5)
        )

        display(
            top5
        )


show_top_changes(
    primary_per_class_df,
    "PRIMARY VALIDATION ANALYSIS"
)

show_top_changes(
    expanded_per_class_df,
    "EXPANDED ROBUSTNESS ANALYSIS"
)


PRIMARY VALIDATION ANALYSIS

FOG


,class_name,fog_normal_iou,fog_adverse_iou,fog_change
11,person,0.440765,0.159951,0.280813
2,building,0.868934,0.813972,0.054962
4,fence,0.484903,0.456511,0.028393
10,sky,0.966049,0.964804,0.001244
8,vegetation,0.857924,0.856692,0.001232



RAIN


,class_name,rain_normal_iou,rain_adverse_iou,rain_change
17,motorcycle,0.646082,0.239886,0.406196
12,rider,0.358480,0.000000,0.358480
18,bicycle,0.716151,0.464595,0.251555
1,sidewalk,0.506580,0.333073,0.173507
4,fence,0.425606,0.274598,0.151009



SNOW


,class_name,snow_normal_iou,snow_adverse_iou,snow_change
9,terrain,0.416034,0.067376,0.348658
12,rider,0.345540,0.000000,0.345540
3,wall,0.479002,0.287206,0.191796
1,sidewalk,0.431983,0.264574,0.167409
16,train,0.704080,0.562962,0.141118


... (6 outputs left out)


SNOW


,class_name,snow_normal_iou,snow_adverse_iou,snow_change
9,terrain,0.561904,0.071491,0.490413
15,bus,0.686887,0.294505,0.392381
18,bicycle,0.533973,0.293227,0.240746
1,sidewalk,0.473491,0.280101,0.193389
14,truck,0.631037,0.443372,0.187664



NIGHT


,class_name,night_normal_iou,night_adverse_iou,night_change
10,sky,0.950883,0.000111,0.950772
16,train,0.836900,0.534026,0.302874
15,bus,0.574800,0.310897,0.263903
12,rider,0.411066,0.181800,0.229266
9,terrain,0.464162,0.253448,0.210714


Save summaries, per-class results, and class-support tables

In [ ]:
primary_summary_df.to_csv(
    MATCHED_RESULTS_DIR
    / "matched_condition_summary_primary_203.csv",
    index=False
)

expanded_summary_df.to_csv(
    MATCHED_RESULTS_DIR
    / "matched_condition_summary_expanded_1003.csv",
    index=False
)

primary_per_class_df.to_csv(
    MATCHED_RESULTS_DIR
    / "matched_per_class_primary_203.csv",
    index=False
)

expanded_per_class_df.to_csv(
    MATCHED_RESULTS_DIR
    / "matched_per_class_expanded_1003.csv",
    index=False
)

primary_support_df.to_csv(
    MATCHED_RESULTS_DIR
    / "matched_class_support_primary_203.csv",
    index=False
)

expanded_support_df.to_csv(
    MATCHED_RESULTS_DIR
    / "matched_class_support_expanded_1003.csv",
    index=False
)

print(
    "Matched analysis tables saved."
)

Matched analysis tables saved.


Save aggregate confusion matrices

In [ ]:
for analysis_name, results in [
    (
        "primary_203",
        primary_results
    ),
    (
        "expanded_1003",
        expanded_results
    ),
]:

    for condition in conditions:

        np.save(
            MATCHED_RESULTS_DIR
            / (
                f"{analysis_name}"
                f"__{condition}"
                f"__normal_confusion.npy"
            ),

            results[
                condition
            ][
                "ref_confusion"
            ]
        )

        np.save(
            MATCHED_RESULTS_DIR
            / (
                f"{analysis_name}"
                f"__{condition}"
                f"__adverse_confusion.npy"
            ),

            results[
                condition
            ][
                "adverse_confusion"
            ]
        )

print(
    "Aggregate confusion matrices saved."
)

Aggregate confusion matrices saved.
